# 01 - Integración y limpieza: construir la tabla "Cliente 360"

Este notebook aplica las 8 reglas de limpieza que se definieron y justificaron en el
notebook `00_diagnostico_y_toma_de_decisiones.ipynb`, y junta las 7 fuentes en una sola
tabla: **una fila por cliente, con toda su información reunida** (Cliente 360).

Esa tabla final es la que van a usar los notebooks siguientes (segmentación, modelo de
propensión, modelo de monto potencial).

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.utils import cargar_tabla, ultimo_saldo_por_cliente

pd.set_option('display.width', 120)
pd.options.display.float_format = '{:,.2f}'.format

## 1. Cargar y limpiar la tabla de clientes

Aquí se aplican 4 de las 8 reglas: quitar clientes repetidos, tratar los vacíos de
`desc_tipo_de_vivienda`, quitar las pocas filas con vacíos en variables financieras, y
quitar las filas con ingresos/egresos imposibles.

In [2]:
clientes = cargar_tabla('../02_Datos/clientes/clientes.db', 'clientes')
print('Filas antes de limpiar:', len(clientes))

Filas antes de limpiar: 860231


**Regla 1 — clientes repetidos.** Nos quedamos con la primera fila de cada `numero_id`
repetido.

In [3]:
clientes = clientes.drop_duplicates(subset='numero_id', keep='first')
print('Filas despues de quitar duplicados:', len(clientes))

Filas despues de quitar duplicados: 860223


**Regla 2 — `desc_tipo_de_vivienda` con muchos vacíos.** En vez de adivinar, los vacíos
se marcan con su propia categoría: `"no informa"`. `desc_genero` tiene el mismo problema
en una escala mucho más pequeña (93 vacíos), así que se trata igual, por consistencia.

In [4]:
clientes['desc_tipo_de_vivienda'] = clientes['desc_tipo_de_vivienda'].fillna('NO INFORMA')
clientes['desc_genero'] = clientes['desc_genero'].fillna('no informa')
clientes['desc_tipo_de_vivienda'].value_counts(dropna=False)

desc_tipo_de_vivienda
NO INFORMA    609881
FAMILIAR      129359
PROPIA         75034
ARRENDADA      45949
Name: count, dtype: int64

**Regla 3 — vacíos pequeños en variables financieras.** Son muy pocos casos (~250 sobre
860 mil), así que simplemente se excluyen esas filas en vez de inventar un valor.

In [5]:
columnas_financieras = ['ingresos_mensuales', 'total_egresos_mensuales',
                        'total_activos', 'total_pasivos', 'total_patrimonio']

antes = len(clientes)
clientes = clientes.dropna(subset=columnas_financieras)
print('Filas quitadas por vacios en variables financieras:', antes - len(clientes))

Filas quitadas por vacios en variables financieras: 260


**Regla 4 — valores imposibles en ingresos y egresos mensuales.** Se definió (con
evidencia, ver notebook 00) un límite de 1.000 millones de pesos al mes — nadie tiene
razonablemente un ingreso o gasto mensual por encima de eso. Ese límite **no** se aplica a
`total_activos`, `total_pasivos` ni `total_patrimonio`, porque esas sí pueden ser
legítimamente muy altas para un cliente de alto patrimonio real (no son un error).

In [6]:
umbral_mensual = 1_000_000_000  # mil millones de pesos al mes

antes = len(clientes)
clientes = clientes[
    (clientes['ingresos_mensuales'] <= umbral_mensual) &
    (clientes['total_egresos_mensuales'] <= umbral_mensual)
]
print('Filas quitadas por ingresos/egresos imposibles:', antes - len(clientes))
print('Filas finales en clientes, ya limpio:', len(clientes))

Filas quitadas por ingresos/egresos imposibles: 167
Filas finales en clientes, ya limpio: 859796


## 2. Consolidar el saldo más reciente de cada producto

**Regla 6.** Por cada producto, usamos la función `ultimo_saldo_por_cliente` (definida en
`src/utils.py`) para quedarnos con un solo número por cliente: su saldo más reciente
conocido — así todos los productos quedan medidos con la misma vara, sin importar cada
cuánto se registró cada uno 

In [7]:
saldo_aho_cte = ultimo_saldo_por_cliente('../02_Datos/crean_aho_cte/crean_aho_cte.db', 'crean_aho_cte')
saldo_bolsillos = ultimo_saldo_por_cliente('../02_Datos/crean_bolsillos/crean_bolsillos.db', 'crean_bolsillos')
saldo_fiducuenta = ultimo_saldo_por_cliente('../02_Datos/crean_fiducuenta/crean_fiducuenta.db', 'crean_fiducuenta')
saldo_cdt_inv_virtual = ultimo_saldo_por_cliente('../02_Datos/crean_inv_virtual_cdt/crean_inv_virtual_cdt.db', 'crean_inv_virtual_cdt')
saldo_invesbot = ultimo_saldo_por_cliente('../02_Datos/invesbot/invesbot.db', 'invesbot')

print('Clientes con saldo de ahorro/corriente:', len(saldo_aho_cte))
print('Clientes con saldo de bolsillos:', len(saldo_bolsillos))
print('Clientes con saldo de fiducuenta:', len(saldo_fiducuenta))
print('Clientes con saldo de CDT/inversion virtual:', len(saldo_cdt_inv_virtual))
print('Clientes con saldo de invesbot:', len(saldo_invesbot))

Clientes con saldo de ahorro/corriente: 475719
Clientes con saldo de bolsillos: 260714
Clientes con saldo de fiducuenta: 181021
Clientes con saldo de CDT/inversion virtual: 84104
Clientes con saldo de invesbot: 5214


**Nota sobre `crean_inv_virtual_cdt`:** esta tabla trae dos productos juntos (CDT e
Inversión Virtual, ver la columna `producto`). Los tratamos como una sola familia de
producto (igual que ya se hace con ahorro/corriente en `crean_aho_cte`), porque el mismo
enunciado de la prueba los agrupa en un solo punto ("CDT e Inversión Virtual"). El error de
codificación que se vio en el notebook 00 (`INVERSI�N VIRTUAL`) no llega a afectar esta
tabla final, porque aquí solo usamos la columna `saldo`, no el texto de `producto`.

## 3. Traer el estimador de ingreso

Esta tabla ya viene con un solo valor por cliente, así que no necesita el mismo
tratamiento de "último saldo".

In [8]:
estimador = cargar_tabla('../02_Datos/estimador_ing/estimador_ing.db', 'estimador_ing')
estimador = estimador.set_index('numero_id')['estimador_ingreso']
print('Clientes con estimador de ingreso:', len(estimador))

Clientes con estimador de ingreso: 745792


## 4. Construir la tabla Cliente 360

**Regla 8.** Ya se validó en el notebook 00 que `numero_id` es consistente entre todas las
tablas (100%), así que se usa directamente como llave para unir todo.

**Regla 7.** Si un cliente no aparece en una tabla de producto, no sabemos con certeza si
es porque no lo tiene o porque no salió en la muestra — lo tratamos igual en ambos casos:
saldo en 0 y una marca de "no tiene evidencia del producto". Por eso guardamos primero la
marca (`tiene_...`) antes de rellenar los saldos vacíos con 0 — así no perdemos la
diferencia entre "sabemos que tiene 0" y "no tenemos información".

In [9]:
cliente_360 = clientes.set_index('numero_id').copy()

productos = {
    'aho_cte': saldo_aho_cte,
    'bolsillos': saldo_bolsillos,
    'fiducuenta': saldo_fiducuenta,
    'cdt_inv_virtual': saldo_cdt_inv_virtual,
    'invesbot': saldo_invesbot,
}

for nombre, serie_saldo in productos.items():
    columna_saldo = f'saldo_{nombre}'
    columna_tiene = f'tiene_{nombre}'

    cliente_360[columna_saldo] = serie_saldo
    cliente_360[columna_tiene] = cliente_360[columna_saldo].notna()
    cliente_360[columna_saldo] = cliente_360[columna_saldo].fillna(0)

cliente_360['estimador_ingreso'] = estimador

cliente_360 = cliente_360.reset_index()
cliente_360.head()

C:\Users\Samuel Murillo\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\indexes\base.py:7866: RuntimeWarning: overflow encountered in scalar subtract
  diff = np_sequence[1] - np_sequence[0]


,numero_id,grupo_edad,desc_genero,desc_segmento,desc_tipo_de_vivienda,ingresos_mensuales,total_egresos_mensuales,total_activos,total_pasivos,total_patrimonio,...,tiene_aho_cte,saldo_bolsillos,tiene_bolsillos,saldo_fiducuenta,tiene_fiducuenta,saldo_cdt_inv_virtual,tiene_cdt_inv_virtual,saldo_invesbot,tiene_invesbot,estimador_ingreso
0,8805210490649048784,65+,masculino,preferencial,NO INFORMA,"29,239,444.00","30,000,000.00","145,047,043,000.00","6,356,702,000.00","105,422,323.00",...,False,0.00,False,0.00,False,0.00,False,0.00,False,NaN
1,-5723572980375902369,50-65,masculino,preferencial,PROPIA,"31,024,544.00","10,000,000.00","1,133,345,000.00","83,072,000.00","1,050,273,000.00",...,False,0.00,False,0.00,False,0.00,False,0.00,False,NaN
2,-8245474570363424359,65+,masculino,preferencial,PROPIA,"2,834,000.00","500,000.00","1,073,787,017.00",0.00,"343,000,000.00",...,False,0.00,False,0.00,False,"450,267,852.65",True,0.00,False,NaN
3,-7840506796880772723,65+,femenino,preferencial,PROPIA,"28,035,850.00",0.00,"175,000,000.00",0.00,"55,000,000.00",...,False,0.00,False,0.00,False,0.00,False,0.00,False,NaN
4,5309731180094827430,36-49,femenino,preferencial,FAMILIAR,"3,846,205.00","2,500,000.00","758,000,000.00",0.00,"758,000,000.00",...,False,0.00,False,0.00,False,0.00,False,0.00,False,NaN


## 5. Revisión final

Antes de guardar, una verificación rápida: que no haya quedado ningún `numero_id`
repetido, que el número de columnas tenga sentido, y que los vacíos que queden sean solo
los esperados (`estimador_ingreso`, que sabemos que no cubre al 100% de los clientes).

In [10]:
print('Filas en Cliente 360:', len(cliente_360))
print('numero_id repetidos:', cliente_360['numero_id'].duplicated().sum())
print()
print('Columnas finales:', list(cliente_360.columns))
print()
print('Vacios restantes por columna:')
print(cliente_360.isnull().sum())

Filas en Cliente 360: 859796
numero_id repetidos: 0

Columnas finales: ['numero_id', 'grupo_edad', 'desc_genero', 'desc_segmento', 'desc_tipo_de_vivienda', 'ingresos_mensuales', 'total_egresos_mensuales', 'total_activos', 'total_pasivos', 'total_patrimonio', 'saldo_aho_cte', 'tiene_aho_cte', 'saldo_bolsillos', 'tiene_bolsillos', 'saldo_fiducuenta', 'tiene_fiducuenta', 'saldo_cdt_inv_virtual', 'tiene_cdt_inv_virtual', 'saldo_invesbot', 'tiene_invesbot', 'estimador_ingreso']

Vacios restantes por columna:
numero_id                       0
grupo_edad                      0
desc_genero                     0
desc_segmento                   0
desc_tipo_de_vivienda           0
ingresos_mensuales              0
total_egresos_mensuales         0
total_activos                   0
total_pasivos                   0
total_patrimonio                0
saldo_aho_cte                   0
tiene_aho_cte                   0
saldo_bolsillos                 0
tiene_bolsillos                 0
saldo_fiducuent

## 6. Guardar el resultado

Esta tabla queda disponible para los notebooks siguientes en `04_Resultados/cliente_360.csv`.

In [11]:
cliente_360.to_csv('../04_Resultados/cliente_360.csv', index=False)
print('Guardado:', len(cliente_360), 'clientes,', len(cliente_360.columns), 'columnas')

Guardado: 859796 clientes, 21 columnas
